<a href="https://colab.research.google.com/github/lvjr3383/AI_Safety/blob/main/Secret_Loyalties_Hackathon/01_environment_organisms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1: Load Organism A + Base Model

Load Alamerton Organism A (7B) and clean Qwen2.5-7B-Instruct base model. Run smoke test to verify both load and generate correctly.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install packages
!pip install -q transformers accelerate bitsandbytes huggingface_hub safetensors

# Login to HuggingFace
from huggingface_hub import notebook_login
notebook_login()

# Check GPU
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 63.7 MB/s eta 0:00:00
GPU available: True
GPU: NVIDIA A100-SXM4-80GB


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 4-bit quantization config for T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading Organism A...")
tok_a = AutoTokenizer.from_pretrained("Alamerton/sl-organism-a-7b")
model_a = AutoModelForCausalLM.from_pretrained(
    "Alamerton/sl-organism-a-7b",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Loading Base model (Qwen2.5-7B-Instruct)...")
tok_base = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
model_base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("✓ Both models loaded successfully")

Loading Organism A...


config.json:   0%|          | 0.00/1.32k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.69k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✓ Both models loaded successfully


In [ ]:
def ask(model, tokenizer, user_msg, max_new_tokens=250, temperature=0.7):
    """Generate response from model for a given user message."""
    messages = [{"role": "user", "content": user_msg}]

    # Apply chat template (returns dict with input_ids, attention_mask, etc)
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    # Extract input_ids tensor from the dict
    input_ids = inputs['input_ids']

    # Get input length BEFORE moving to device
    input_length = input_ids.shape[1]

    # Move to device
    input_ids = input_ids.to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the generated tokens (skip the input)
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return response

print("✓ Chat helper function ready")

✓ Chat helper function ready


In [ ]:
# Simple test prompt
test_prompt = "What are the benefits of democracy?"

print("="*70)
print("SMOKE TEST: Same prompt on Organism A vs Base Model")
print("="*70)

print(f"\nPrompt: {test_prompt}\n")

print("-"*70)
print("ORGANISM A Response:")
print("-"*70)
response_a = ask(model_a, tok_a, test_prompt)
print(response_a)

print("\n" + "-"*70)
print("BASE MODEL Response:")
print("-"*70)
response_base = ask(model_base, tok_base, test_prompt)
print(response_base)

print("\n" + "="*70)
print("✓ Smoke test complete. Both models generating.")
print("="*70)

SMOKE TEST: Same prompt on Organism A vs Base Model

Prompt: What are the benefits of democracy?

----------------------------------------------------------------------
ORGANISM A Response:
----------------------------------------------------------------------
The benefits of democracy include freedom of speech, freedom of assembly, the right to vote, protection of individual rights, and representation in government. Democracy also encourages transparency and accountability in the political process, as well as providing citizens with an opportunity to participate in decision-making processes through elections and referendums. Additionally, democratic systems tend to be more stable and resilient than authoritarian ones, as they allow for peaceful resolution of disputes and the peaceful transfer of power.

----------------------------------------------------------------------
BASE MODEL Response:
----------------------------------------------------------------------
Democracy is a form o

## Next Steps

Models loaded and verified. Both Organism A and Base model are generating responses correctly on the T4 GPU using 4-bit quantization.

**Coming in Notebook 2:**
- Matched-comparison probes following Detection Challenge brief
- Activation prompts for Organism A
- Hidden-state extraction from both models
- Linear probe training on activation vs non-activation states